# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and analyzing the FAIR^2 dataset using the `mlcroissant` library. All dataset entities (record sets, fields, columns) are referenced by their `@id`, which uniquely identifies each data element under the Croissant schema.

### Dataset Source
The dataset source is specified via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load metadata and dataset structure
dataset = mlc.Dataset(croissant_url)

# Access metadata (as a single object)
metadata = dataset.metadata

# Display dataset title and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their IDs as defined in the Croissant schema.

In [ ]:
# List available record sets and their @ids
record_sets = list(dataset.record_sets.keys())
print('Record Sets (@id):')
for rs_id in record_sets:
    rs = dataset.record_sets[rs_id]
    print(f"- @id: {rs_id}")
    print(f"  name: {rs.name if hasattr(rs, 'name') else 'N/A'}")
    print(f"  description: {rs.description if hasattr(rs, 'description') else 'N/A'}")
    # List fields in this record set
    if hasattr(rs, 'fields'):
        print("  Fields (@id):")
        for fld in rs.fields:
            print(f"    * @id: {fld['@id']}, name: {fld['name']}" if 'name' in fld else f"    * @id: {fld['@id']}")
    # List columns in this record set
    if hasattr(rs, 'columns'):
        print("  Columns (@id):")
        for col in rs.columns:
            print(f"    * @id: {col['@id']}, name: {col['name']}" if 'name' in col else f"    * @id: {col['@id']}")

# Preview a few records from each record set
print("\nPreview records:")
for rs_id in record_sets:
    print(f"\nRecords from record set @id: {rs_id}")
    for i, record in enumerate(dataset.records(record_set=rs_id)):
        if i > 2:
            break
        pprint.pprint(record)

## 3. Data Extraction
Load data from each record set into a DataFrame for further analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract all available record sets into DataFrames by their @id
dataframes = {}

for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df

# Display columns available in each DataFrame (by field @id)
for rs_id, df in dataframes.items():
    print(f"\nColumns in record set @id: {rs_id}: {df.columns.tolist()}")
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

The following example uses typical fields (with their `@id`) from the first record set loaded. You may adjust the field IDs and operations based on the actual field types present.

In [ ]:
# Choose the first record set and examine its numeric/categorical fields
if len(dataframes) > 0:
    first_rs_id = list(dataframes.keys())[0]
    df = dataframes[first_rs_id]
    print(f"Working with record set @id: {first_rs_id}")

    # Identify potential numeric field (@id)
    numeric_candidates = [col for col in df.columns if df[col].dtype in [int, float] or df[col].str.replace('.', '').str.isdigit().all()]
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
        print(f"Selected numeric field @id: {numeric_field_id}")
        # Convert field to numeric if needed
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].mean() if not df[numeric_field_id].isnull().all() else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by a categorical field (@id), if present
        group_candidates = [col for col in df.columns if df[col].dtype == object and col != numeric_field_id]
        if group_candidates:
            group_field_id = group_candidates[0]
            print(f"\nGrouping by field @id: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())
else:
    print("No dataframes found to perform EDA.")

## 5. Visualization
Visualize distributions or relationships between selected fields in the dataset. Adjust field IDs as needed.

In [ ]:
# Basic visualization using matplotlib or seaborn
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes) > 0 and numeric_candidates:
    # Histogram of the numeric field
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of field @id: {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If grouping field exists, show boxplot
    if group_candidates:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by group field @id: {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated dataset loading, record set discovery, extraction, processing, and basic visualization using the `mlcroissant` library. All entity references are made using their Croissant `@id`.

This FAIR^2 dataset is highly structured and includes clinical features, hormone levels, imaging, and genetic variant data for rare disease research. Please review the individual record set fields (`@id`) for further detailed analysis as needed.